# Conceitos do Projeto PNCP

Este notebook explica, com **dados de brinquedo**, os conceitos por trás do projeto avaliativo do Módulo 2.

- **Não** lê os CSVs de `data/raw` e **não** substitui o kit (`src/limpeza.py`, `src/montar_tabelas.py`, `src/carregar.py`).
- Cada seção mostra **o problema**, **a ideia** e **uma pergunta para você pensar** antes de completar o seu código.
- Quando o problema é idêntico a uma lacuna `____` do kit, o exemplo usa um caso **parecido** (CEP no lugar de CNPJ, por exemplo): a técnica é a mesma, a resposta do kit é sua.
- No fim, duas células **conferem o seu trabalho**: uma testa as funções do seu `src/limpeza.py`, a outra compara os seus arquivos de `data/processed/` com a prova real.

| # | Seção | Desafio / regra do enunciado |
|---|---|---|
| 1 | Por que ler tudo como texto | base de tudo |
| 2 | Auditar: descobrir os formatos | Sprint 1 |
| 3 | Nulos disfarçados | desafio 4 |
| 4 | Identificadores (CNPJ/CPF) | desafio 1 |
| 5 | Separadores numéricos | desafio 2, regra 12 |
| 6 | Encoding quebrado | desafio 5 |
| 7 | Um vencedor por item | regras 2 e 3 |
| 8 | Valores e rejeição | regras 4 e 5 |
| 9 | Comparável e economia | regra 6 |
| 10 | Cronologia e ciclo | regra 7 |
| 11 | Perfil do gasto e `\b` | regra 8 |
| 12 | Star Schema e chaves substitutas | seção 4.3 |
| 13 | Conferência com a prova real | seção 4.7 / anexo 8 |

In [1]:
import io
import re
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)

# Acha a raiz do projeto (a pasta que tem src/ e data/), subindo a partir de onde o notebook esta
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "data").is_dir()), None)
print("Raiz do projeto encontrada:", RAIZ.name if RAIZ else "NAO (abra o notebook dentro da pasta do projeto)")

Raiz do projeto encontrada: Projeto Final - Módulo 2


## 1. Por que ler tudo como texto (`dtype=str`)

O pandas tenta "adivinhar" o tipo de cada coluna. Com dados sujos, essa adivinhação **destrói informação** antes de você conseguir tratá-la:

- um CNPJ que começa com zero vira número e **perde o zero**;
- `"N/A"` e `"NULL"` viram `NaN` e você não sabe mais o que estava escrito;
- um valor `"1.234,56"` fica como texto numa coluna e número em outra.

Por isso o kit lê com `dtype=str, keep_default_na=False`: **nada muda até você decidir**.

In [2]:
csv_bruto = io.StringIO(
    "cnpj,valor,situacao\n"
    "01234567000189,\"1.234,56\",N/A\n"
    "98765432000110,99.5,NULL\n"
)

adivinhado = pd.read_csv(csv_bruto)
csv_bruto.seek(0)
como_texto = pd.read_csv(csv_bruto, dtype=str, keep_default_na=False)

print("Leitura padrao (o pandas adivinha):")
print(adivinhado, "\n", adivinhado.dtypes.to_dict(), "\n")
print("Leitura como texto (nada muda):")
print(como_texto, "\n", como_texto.dtypes.to_dict())

Leitura padrao (o pandas adivinha):
             cnpj     valor  situacao
0   1234567000189  1.234,56       NaN
1  98765432000110      99.5       NaN 
 {'cnpj': dtype('int64'), 'valor': <StringDtype(storage='python', na_value=nan)>, 'situacao': dtype('float64')} 

Leitura como texto (nada muda):
             cnpj     valor situacao
0  01234567000189  1.234,56      N/A
1  98765432000110      99.5     NULL 
 {'cnpj': <StringDtype(storage='python', na_value=nan)>, 'valor': <StringDtype(storage='python', na_value=nan)>, 'situacao': <StringDtype(storage='python', na_value=nan)>}


> **Para pensar:** na leitura padrão, o que aconteceu com o primeiro CNPJ? E com `N/A` e `NULL`? Você conseguiria contar quantos `N/A` existiam?

## 2. Auditar: descobrir os formatos antes de limpar

Antes de escrever qualquer regra, **descubra o que existe** na coluna. Uma técnica simples é transformar cada valor no seu "formato": todo dígito vira `9` e toda letra vira `A`. Depois é só contar os formatos.

Isso responde às perguntas da seção 7.4 do enunciado ("que formatos de dinheiro, data e CNPJ aparecem?") e dá os números para o **dicionário de inconsistências** do README.

In [3]:
def formato(valor):
    # Troca digitos por 9 e letras por A, mantendo pontuacao e espacos: '12/03/2025' -> '99/99/9999'
    valor = re.sub(r"\d", "9", str(valor))
    return re.sub(r"[A-Za-zÀ-ÿ]", "A", valor)


datas_brinquedo = pd.Series(["2025-03-12", "12/03/2025", "2025-03-12 14:30:00", "12/03/2025", "N/A", "", "20250312"])
print(datas_brinquedo.map(formato).value_counts().to_string())

99/99/9999             2
9999-99-99             1
9999-99-99 99:99:99    1
A/A                    1
                       1
99999999               1


> **Para pensar:** rode essa mesma ideia nas colunas de data, dinheiro e documento dos CSVs reais (num script ou num notebook seu). Quantos formatos aparecem em `data_resultado`? E em `ni_fornecedor`? Anote no README: isso é a Sprint 1.

## 3. Nulos disfarçados

Sistemas legados preenchem campos vazios com textos que **parecem** valores: `"N/A"`, `"NULL"`, `"-"`, `"SEM INFORMAÇÃO"`, `"NÃO SE APLICA"`. Se você não tratar, eles viram um "órgão" chamado `N/A` ou um fornecedor chamado `-`.

A regra do enunciado tem uma pegadinha: **`"0"` só é nulo em campo de texto.** Numa quantidade ou num valor, zero é um número (e quem decide o que fazer com ele é a regra de rejeição).

In [4]:
NULOS = {"", "N/A", "NULL", "-", "SEM INFORMAÇÃO", "NÃO SE APLICA"}

nome_fornecedor = pd.Series(["EMPRESA ALFA", " n/a ", "-", "0", "Sem Informação", "EMPRESA BETA"])
quantidade = pd.Series(["10", "0", "N/A", "5"])

# Comparar sempre depois de tirar espacos e padronizar maiusculas
eh_nulo_texto = nome_fornecedor.str.strip().str.upper().isin(NULOS | {"0"})   # em texto, "0" tambem e nulo
eh_nulo_numero = quantidade.str.strip().str.upper().isin(NULOS)               # em numero, "0" NAO e nulo

print(pd.DataFrame({"nome_fornecedor": nome_fornecedor, "nulo?": eh_nulo_texto}), "\n")
print(pd.DataFrame({"quantidade": quantidade, "nulo?": eh_nulo_numero}))

  nome_fornecedor  nulo?
0    EMPRESA ALFA  False
1            n/a    True
2               -   True
3               0   True
4  Sem Informação   True
5    EMPRESA BETA  False 

  quantidade  nulo?
0         10  False
1          0  False
2        N/A   True
3          5  False


> **Para pensar:** por que comparar depois de `.strip().upper()`? O que aconteceria com `" n/a "` sem isso? No kit, onde está o parâmetro que diz se `"0"` é nulo?

## 4. Identificadores: máscara e zero perdido

Um mesmo documento pode chegar de vários jeitos: com máscara (`12.345.678/0001-90`), sem máscara, ou **sem o zero da esquerda** (alguém abriu no Excel, que transformou em número). Se você não padronizar, **o mesmo fornecedor vira dois** e a contagem de fornecedores explode.

A técnica tem dois passos: **ficar só com os dígitos** e **completar com zeros à esquerda** até o tamanho certo. O tamanho depende do tipo do documento: CNPJ tem 14 dígitos, CPF tem 11.

Abaixo, a mesma técnica num caso parecido: o **CEP** (8 dígitos).

In [5]:
ceps = pd.Series(["88.010-000", "88010000", "8010-000", "1310100", " 01310-100 "])

so_digitos = ceps.map(lambda v: re.sub(r"\D", "", v))   # \D = tudo que NAO e digito
cep_limpo = so_digitos.str.zfill(8)                      # completa com zeros a esquerda ate 8

print(pd.DataFrame({"original": ceps, "so digitos": so_digitos, "limpo": cep_limpo}))
print("\nCEPs distintos antes:", ceps.nunique(), "| depois:", cep_limpo.nunique())

      original so digitos     limpo
0   88.010-000   88010000  88010000
1     88010000   88010000  88010000
2     8010-000    8010000  08010000
3      1310100    1310100  01310100
4   01310-100    01310100  01310100

CEPs distintos antes: 5 | depois: 3


> **Para pensar:** no projeto, quem diz se o documento tem 14 ou 11 dígitos? (Dica: há uma coluna `tipo_pessoa` com PJ, PF e PE.) E o que fazer com um estrangeiro (PE), que não tem CNPJ brasileiro?

## 5. Separadores numéricos (o erro que mais derruba o volume)

No Brasil escrevemos `1.234,56`; em outros sistemas, `1,234.56`. Nesta base vêm **os dois**, e mais `R$`, espaços invisíveis e negativos entre parênteses. A regra do enunciado decide os casos ambíguos:

| Texto na base | Vale | Por quê |
|---|---|---|
| `0,754` | 0,754 | um separador **sozinho** é sempre decimal |
| `746.815` | 746,815 | idem (não é 746 mil!) |
| `1.234.567` | 1.234.567 | separador **repetido** é milhar |
| `1.234,56` | 1.234,56 | com os dois símbolos, o **último** é o decimal |
| `1,234.56` | 1.234,56 | idem |
| `R$ 1.234,56` | 1.234,56 | tirar o `R$` |
| `(500,00)` | −500,00 | parênteses = negativo |

Se você ler `746.815` como 746 mil, o volume homologado sobe de **R$ 8 bi para R$ 31 bi** (seção 4.7 do enunciado).

As ferramentas úteis para decidir: contar quantas vezes cada símbolo aparece e ver qual vem por último.

In [6]:
for texto in ["0,754", "746.815", "1.234.567", "1.234,56", "1,234.56"]:
    pontos, virgulas = texto.count("."), texto.count(",")
    ultimo = max(texto.rfind("."), texto.rfind(","))
    print(f"{texto:>10}  pontos={pontos} virgulas={virgulas}  ultimo separador='{texto[ultimo]}'")

     0,754  pontos=0 virgulas=1  ultimo separador=','
   746.815  pontos=1 virgulas=0  ultimo separador='.'
 1.234.567  pontos=2 virgulas=0  ultimo separador='.'
  1.234,56  pontos=1 virgulas=1  ultimo separador=','
  1,234.56  pontos=1 virgulas=1  ultimo separador='.'


> **Para pensar:** com `pontos`, `virgulas` e o último separador, escreva (no papel) a sequência de decisões da sua função `converter_numero`. Qual caso você testa primeiro? Depois implemente em `src/limpeza.py` e use a célula da seção 13 para testar com estes exemplos.

Um detalhe que engana: **espaços invisíveis**. O caractere `\xa0` (espaço não separável) parece um espaço, mas `strip()` comum de alguns métodos não o remove, e `float()` falha.

In [7]:
invisivel = "\xa01.234,56"
print(repr(invisivel), "| parece:", invisivel, "| tamanho:", len(invisivel))
print("Contem espaco comum?", " " in invisivel, "| contem \\xa0?", "\xa0" in invisivel)

'\xa01.234,56' | parece:  1.234,56 | tamanho: 9
Contem espaco comum? False | contem \xa0? True


## 6. Encoding quebrado (`Ã§` no lugar de `ç`)

Quando um texto gravado em **UTF-8** é lido como **Latin-1**, cada letra acentuada vira dois caracteres estranhos. É o "mojibake". Como é um erro **reversível**, dá para desfazer fazendo o caminho inverso.

In [8]:
original = "Aquisição de cadeiras para o órgão"
quebrado = original.encode("utf-8").decode("latin-1")
consertado = quebrado.encode("latin-1").decode("utf-8")

print("original  :", original)
print("quebrado  :", quebrado)
print("consertado:", consertado)

original  : Aquisição de cadeiras para o órgão
quebrado  : AquisiÃ§Ã£o de cadeiras para o Ã³rgÃ£o
consertado: Aquisição de cadeiras para o órgão


> **Para pensar:** e se o texto **já estiver certo**? O que acontece se você aplicar o conserto num texto sem problema? (Teste trocando `quebrado` por `original` na última linha.) Por isso a função do kit só conserta quando encontra sinais de quebra, como `Ã`.

## 7. Um vencedor por item: a ordem das operações importa

Em Registro de Preços, um item tem vários classificados. A regra:

1. **Primeiro** descarte os resultados cancelados.
2. **Depois**, entre os que sobraram, fique com o menor `ordem_classificacao_srp` (vazio vale 1).
3. Empate? Menor `sequencial_resultado`, **lido como número** (2 vem antes de 10).

Veja o que acontece quando a ordem é invertida:

In [9]:
resultados = pd.DataFrame({
    "item":        ["A", "A", "B", "B", "C", "C"],
    "fornecedor":  ["F1", "F2", "F3", "F4", "F5", "F6"],
    "ordem":       ["1", "2", "1", "2", "", ""],
    "sequencial":  ["1", "2", "1", "2", "10", "2"],
    "cancelado":   [True, False, False, False, False, False],
})
resultados["ordem_num"] = pd.to_numeric(resultados["ordem"], errors="coerce").fillna(1)
resultados["seq_num"] = pd.to_numeric(resultados["sequencial"], errors="coerce")

ordenado = resultados.sort_values(["item", "ordem_num", "seq_num"])

errado = ordenado.drop_duplicates("item")              # pega o 1o colocado...
errado = errado[~errado["cancelado"]]                  # ...e so depois tira os cancelados

certo = ordenado[~ordenado["cancelado"]]               # tira os cancelados primeiro...
certo = certo.drop_duplicates("item")                  # ...e so depois pega o 1o colocado

print("ERRADO:\n", errado[["item", "fornecedor"]].to_string(index=False), "\n")
print("CERTO:\n", certo[["item", "fornecedor"]].to_string(index=False))

ERRADO:
 item fornecedor
   B         F3
   C         F6 

CERTO:
 item fornecedor
   A         F2
   B         F3
   C         F6


> **Para pensar:** no jeito errado, o item A **sumiu**. No projeto, isso faz faltar cerca de 60 itens na fato (6.065 em vez de 6.126). E o item C: por que o vencedor é F6 e não F5? O que aconteceria se `sequencial` fosse ordenado como texto?

## 8. Valores e rejeição: o `NaN` é traiçoeiro

**Valores (regra 5):** total = unitário × quantidade, usando a **quantidade homologada** (ou a do item, se vier vazia ou zero) no estimado **e** no homologado, para comparar maçã com maçã.

**Rejeição (regra 4):** o item sai da fato se tiver dados impossíveis. A armadilha está no `NaN`: **toda comparação com `NaN` dá `False`**. Então `quantidade <= 0` **não pega** a quantidade vazia.

In [10]:
itens = pd.DataFrame({"item": ["A", "B", "C"], "quantidade": [10.0, 0.0, float("nan")]})

itens["qtd <= 0"] = itens["quantidade"] <= 0
itens["~(qtd > 0)"] = ~(itens["quantidade"] > 0)
print(itens)

  item  quantidade  qtd <= 0  ~(qtd > 0)
0    A        10.0     False       False
1    B         0.0      True        True
2    C         NaN     False        True


> **Para pensar:** qual das duas colunas implementa "quantidade **vazia ou** ≤ 0"? Repare que o kit usa exatamente esse truque com o til (`~`).

Outro ponto importante: item **sem resultado** não é "rejeitado". Ele já saiu antes, na regra 3. Por isso os números da prova real são 7.077 itens − 852 sem resultado = 6.225 avaliados, − 99 rejeitados = **6.126 na fato**.

## 9. Comparável e economia: soma ÷ soma, nunca média de percentuais

Um item é **comparável** quando não é Credenciamento **e** a razão homologado ÷ estimado está entre 0,1 e 10. A economia só usa os comparáveis.

Por que **soma ÷ soma**? Porque a média de percentuais dá o mesmo peso a um item de R$ 10 e a um de R$ 10 milhões.

In [11]:
itens = pd.DataFrame({
    "item":       ["caneta", "notebook", "obra", "credenciamento", "erro de digitacao"],
    "modalidade": [6, 6, 4, 12, 6],
    "estimado":   [10.0, 5_000.0, 1_000_000.0, 50_000.0, 100.0],
    "homologado": [2.0, 4_500.0, 900_000.0, 50_000.0, 90_000.0],
})
itens["razao"] = itens["homologado"] / itens["estimado"]
itens["comparavel"] = ((itens["modalidade"] != 12) & itens["razao"].between(0.1, 10)).astype(int)
itens["economia"] = (itens["estimado"] - itens["homologado"]).where(itens["comparavel"] == 1)
print(itens, "\n")

comp = itens[itens["comparavel"] == 1]
media_de_percentuais = (comp["economia"] / comp["estimado"]).mean()
soma_sobre_soma = comp["economia"].sum() / comp["estimado"].sum()
print(f"Media de percentuais: {media_de_percentuais:.2%}   <- a caneta de R$ 10 pesa igual a obra")
print(f"Soma / soma:          {soma_sobre_soma:.2%}   <- cada real pesa igual (o certo)")

                item  modalidade   estimado  homologado  razao  comparavel  economia
0             caneta           6       10.0         2.0    0.2           1       8.0
1           notebook           6     5000.0      4500.0    0.9           1     500.0
2               obra           4  1000000.0    900000.0    0.9           1  100000.0
3     credenciamento          12    50000.0     50000.0    1.0           0       NaN
4  erro de digitacao           6      100.0     90000.0  900.0           0       NaN 

Media de percentuais: 33.33%   <- a caneta de R$ 10 pesa igual a obra
Soma / soma:          10.00%   <- cada real pesa igual (o certo)


> **Para pensar:** o "erro de digitação" (razão 900) ficou de fora da economia, mas **continua na fato** e conta no volume. E o credenciamento: por que a economia dele fica vazia, e não zero? No Power BI, a Economia % de Credenciamento aparece em branco, e isso **é o esperado**.

## 10. Cronologia e ciclo

- Homologação **antes** da publicação é impossível: o item fica sem data de homologação e sem ciclo, **mas continua na fato**.
- Compare **só as datas**, sem hora: homologar no mesmo dia da publicação é válido (ciclo 0).
- A média do ciclo **ignora os vazios** (no pandas, no SQL com `AVG` e no DAX com `AVERAGE`).

In [12]:
datas = pd.DataFrame({
    "item":        ["A", "B", "C", "D"],
    "publicacao":  pd.to_datetime(["2025-03-10 09:00", "2025-03-10 18:00", "2025-03-10 00:00", "2025-03-10 00:00"]),
    "homologacao": pd.to_datetime(["2025-04-09 10:00", "2025-03-10 08:00", "2025-03-01 00:00", None]),
})
pub, hom = datas["publicacao"].dt.normalize(), datas["homologacao"].dt.normalize()   # normalize = zera a hora

datas["valida"] = hom >= pub
datas["dias_ciclo"] = (hom - pub).dt.days.where(datas["valida"])
print(datas, "\n")
print("Ciclo medio (ignora vazios):", datas["dias_ciclo"].mean())
print("Itens sem ciclo:", datas["dias_ciclo"].isna().sum())

  item          publicacao         homologacao  valida  dias_ciclo
0    A 2025-03-10 09:00:00 2025-04-09 10:00:00    True        30.0
1    B 2025-03-10 18:00:00 2025-03-10 08:00:00    True         0.0
2    C 2025-03-10 00:00:00 2025-03-01 00:00:00   False         NaN
3    D 2025-03-10 00:00:00                 NaT   False         NaN 

Ciclo medio (ignora vazios): 15.0
Itens sem ciclo: 2


> **Para pensar:** o item B foi homologado às 08h e publicado às 18h do mesmo dia. Se você comparasse com a hora, ele seria "impossível". Com só a data, tem ciclo 0. Qual comportamento o enunciado pede?

## 11. Perfil do gasto: procure a palavra inteira (`\b`)

Para separar **Obras e engenharia**, o enunciado procura palavras-chave na descrição do item e no objeto da compra. Sem o limite de palavra `\b`, "cobrança" e "dobradiça" contêm "obra".

In [13]:
textos = pd.Series(["REFORMA DO TELHADO", "SERVICO DE COBRANCA", "DOBRADICA DE ACO", "OBRAS DE ACESSIBILIDADE", "MANOBRA DE VEICULOS"])

sem_limite = textos.str.contains(r"OBRA", regex=True)
com_limite = textos.str.contains(r"\b(?:OBRAS?|REFORMA)\b", regex=True)
print(pd.DataFrame({"texto": textos, "sem \\b": sem_limite, "com \\b": com_limite}))

                     texto  sem \b  com \b
0       REFORMA DO TELHADO   False    True
1      SERVICO DE COBRANCA    True   False
2         DOBRADICA DE ACO    True   False
3  OBRAS DE ACESSIBILIDADE    True    True
4      MANOBRA DE VEICULOS    True   False


> **Para pensar:** quais palavras você vai usar? A referência usa quatro: obra, construção, reforma, engenharia. Se escolher outras, o percentual muda um pouco, e tudo bem, desde que você **documente no README**.

## 12. Star Schema e chaves substitutas

O "tabelão" repete o nome do órgão, a UF e a modalidade em **cada item**. O modelo estrela separa:

- **Fato:** o que se mede (valores, quantidades, datas), uma linha por item.
- **Dimensões:** quem, como e onde. Uma linha por valor distinto da **chave natural** (CNPJ, UF, modalidade_id).
- **Chave substituta (`sk_`):** um número 1, 2, 3… que liga a fato à dimensão. Ela é estável, pequena e independe de o CNPJ vir sujo.
- **`sk = -1` "Não informado":** quando falta o dado, a FK aponta para essa linha, e **nenhuma FK fica solta**.

In [14]:
tabelao = pd.DataFrame({
    "item":  [1, 2, 3, 4, 5],
    "uf":    ["SC", "SP", "SC", "RS", None],
    "valor": [100.0, 250.0, 80.0, 40.0, 10.0],
})

# Dimensao: uma linha por chave natural, numerada
dim_uf = tabelao.dropna(subset=["uf"]).drop_duplicates("uf")[["uf"]].reset_index(drop=True)
dim_uf.insert(0, "sk_uf", dim_uf.index + 1)
dim_uf = pd.concat([pd.DataFrame({"sk_uf": [-1], "uf": ["NI"]}), dim_uf], ignore_index=True)   # linha "Nao informado"

# Fato: troca a chave natural pela substituta; o que nao achar vira -1
fato = tabelao.copy()
fato["sk_uf"] = fato["uf"].map(dict(zip(dim_uf["uf"], dim_uf["sk_uf"]))).fillna(-1).astype(int)
fato = fato[["item", "sk_uf", "valor"]]

print("dim_uf:\n", dim_uf.to_string(index=False), "\n")
print("fato:\n", fato.to_string(index=False), "\n")

# Integridade: toda FK da fato precisa existir na dimensao
soltas = ~fato["sk_uf"].isin(dim_uf["sk_uf"])
print("FKs soltas:", soltas.sum())

dim_uf:
  sk_uf uf
    -1 NI
     1 SC
     2 SP
     3 RS 

fato:
  item  sk_uf  valor
    1      1  100.0
    2      2  250.0
    3      1   80.0
    4      3   40.0
    5     -1   10.0 

FKs soltas: 0


> **Para pensar:** no banco, o que garante que uma FK solta **não entra**? (A `FOREIGN KEY` no `01_criar_banco.sql`.) E o que garante que o mesmo CNPJ não aparece duas vezes na dimensão? (`UNIQUE` na chave natural.) Por isso o enunciado chama o banco de **segunda barreira**.

O banco de exemplo em `resposta/exemplo_sql/` mostra esse modelo completo, com dados fictícios, pronto para o Power BI.

## 13. Conferência do seu trabalho

As duas células abaixo **não resolvem nada**: elas testam o que **você** escreveu.

### 13.1 Testar as suas funções de `src/limpeza.py`

Usa os exemplos da seção 5. Enquanto a função tiver `____`, aparece "ainda não implementada". (O autoteste oficial continua sendo `python -m src.limpeza`.)

In [15]:
CASOS_NUMERO = {
    "0,754": 0.754, "746.815": 746.815, "1.234.567": 1234567.0, "1.234,56": 1234.56,
    "1,234.56": 1234.56, "R$ 1.234,56": 1234.56, "(500,00)": -500.0, "\xa01.234,56": 1234.56,
}

if RAIZ is None:
    print("Raiz do projeto nao encontrada: abra o notebook dentro da pasta do projeto.")
else:
    sys.path.insert(0, str(RAIZ))
    import importlib
    from src import limpeza
    importlib.reload(limpeza)   # pega a versao mais recente do arquivo

    for texto, esperado in CASOS_NUMERO.items():
        try:
            obtido = limpeza.converter_numero(texto)
            ok = obtido is not None and abs(obtido - esperado) < 1e-9
            print(f"{'OK ' if ok else 'ERRO'}  {texto!r:>16} -> {obtido!r}   (esperado {esperado})")
        except NameError:
            print("converter_numero ainda nao implementada (tem ____ no codigo).")
            break
        except Exception as erro:
            print(f"ERRO  {texto!r:>16} -> {type(erro).__name__}: {erro}")

converter_numero ainda nao implementada (tem ____ no codigo).


### 13.2 Comparar `data/processed/` com a prova real (anexo 8)

Depois de rodar `python -m src.montar_tabelas`, esta célula lê os seus CSVs e compara com os números de referência. Diferença pequena e justificada é normal; diferença grande indica erro (veja a tabela da seção 4.7 do enunciado).

In [16]:
PROVA_REAL = {
    "itens na fato": 6126,
    "contratacoes distintas": 1962,
    "fornecedores": 2652,
    "unidades compradoras": 363,
    "volume homologado (R$)": 8_044_498_805.74,
    "estimado comparavel (R$)": 6_867_663_469.91,
    "itens comparaveis": 5457,
    "economia (%)": 14.46,
    "itens sem ciclo": 230,
    "ciclo medio (dias)": 48.1,
    "ciclo maximo (dias)": 611,
}

pasta = RAIZ / "data" / "processed" if RAIZ else None
if pasta is None or not (pasta / "fato_item_contratado.csv").exists():
    print("data/processed/fato_item_contratado.csv ainda nao existe: rode `python -m src.montar_tabelas` primeiro.")
else:
    f = pd.read_csv(pasta / "fato_item_contratado.csv")
    comp = f["comparavel"] == 1
    meu = {
        "itens na fato": len(f),
        "contratacoes distintas": f["numero_controle_pncp"].nunique(),
        "fornecedores": len(pd.read_csv(pasta / "dim_fornecedor.csv", dtype=str)),
        "unidades compradoras": len(pd.read_csv(pasta / "dim_orgao.csv", dtype=str)),
        "volume homologado (R$)": round(f["valor_total_homologado"].sum(), 2),
        "estimado comparavel (R$)": round(f.loc[comp, "valor_total_estimado"].sum(), 2),
        "itens comparaveis": int(comp.sum()),
        "economia (%)": round(100 * f.loc[comp, "economia_valor"].sum() / f.loc[comp, "valor_total_estimado"].sum(), 2),
        "itens sem ciclo": int(f["dias_ciclo"].isna().sum()),
        "ciclo medio (dias)": round(f["dias_ciclo"].mean(), 1),
        "ciclo maximo (dias)": f["dias_ciclo"].max(),
    }
    conferencia = pd.DataFrame({"esperado": PROVA_REAL, "meu": meu})
    conferencia["bate?"] = [abs(m - e) <= max(1, abs(e) * 0.0001) for e, m in zip(conferencia["esperado"], conferencia["meu"])]
    print(conferencia.to_string())

data/processed/fato_item_contratado.csv ainda nao existe: rode `python -m src.montar_tabelas` primeiro.


## 14. Onde este notebook entra na entrega

O enunciado pede que o **pipeline** seja feito em scripts (`src/`), executáveis de ponta a ponta. Um notebook é **opcional** e serve para **explorar e auditar**, não para substituir os scripts.

Se você criar um notebook seu (por exemplo, a auditoria da Sprint 1), coloque-o numa pasta própria:

```text
notebooks/
└── 01_auditoria_dados_brutos.ipynb    ← exploração; o pipeline continua em src/
```

Antes de versionar um notebook:

- rode **Reiniciar e executar tudo**, para as saídas ficarem na ordem certa;
- confira se não ficou caminho fixo da sua máquina (`C:\Users\...`) nem senha;
- não salve saídas gigantes (um `print` de 15 mil linhas deixa o arquivo pesado no GitHub).

O guia completo de organização está em [`organizacao_do_projeto.md`](../organizacao_do_projeto.md), e o checklist dos critérios em [`esqueleto_criterios.md`](../esqueleto_criterios.md).